# SpaceX Falcon 9 — Exploratory Data Analysis with SQL

**Author:** Piyu

We load the wrangled dataset into a SQLite database and use SQL queries to answer specific business questions about the launch data. (SQLite is used here in place of PostgreSQL for portability — same SQL logic applies.)

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/spacex.db')
df = pd.read_csv('../data/dataset_wrangled.csv')
df.to_sql('SPACEXTABLE', conn, if_exists='replace', index=False)
print("Loaded", len(df), "rows into SPACEXTABLE")

Loaded 101 rows into SPACEXTABLE


## 1. Unique launch sites

In [2]:
pd.read_sql_query("SELECT DISTINCT Launch_Site FROM SPACEXTABLE", conn)

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


## 2. Launch records where site begins with 'CCA'

In [3]:
pd.read_sql_query("""
SELECT * FROM SPACEXTABLE
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5
""", conn)

,FlightNumber,Date,Booster_Version,Launch_Site,Payload,PayloadMass,Orbit,Customer,Mission_Outcome,Landing _Outcome,Class
0,1,2010-04-06,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute),0
1,2,2010-08-12,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),0
2,3,2012-05-22,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt,0
3,4,2012-08-10,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt,0
4,5,2013-01-03,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt,0


## 3. Total payload mass carried by NASA (CRS) boosters

In [4]:
pd.read_sql_query("""
SELECT SUM(PayloadMass) AS Total_Payload_Mass_KG
FROM SPACEXTABLE
WHERE Customer LIKE '%NASA (CRS)%'
""", conn)

,Total_Payload_Mass_KG
0,48213


## 4. Average payload mass carried by booster version F9 v1.1

In [5]:
pd.read_sql_query("""
SELECT AVG(PayloadMass) AS Avg_Payload_Mass_KG
FROM SPACEXTABLE
WHERE Booster_Version LIKE 'F9 v1.1%'
""", conn)

,Avg_Payload_Mass_KG
0,2534.666667


## 5. Date of first successful ground pad landing

In [6]:
pd.read_sql_query("""
SELECT MIN(Date) AS First_Ground_Pad_Success
FROM SPACEXTABLE
WHERE \"Landing _Outcome\" = 'Success (ground pad)'
""", conn)

,First_Ground_Pad_Success
0,2015-12-22


## 6. Boosters with successful drone-ship landing, payload between 4000 and 6000 kg

In [7]:
pd.read_sql_query("""
SELECT Booster_Version, PayloadMass, \"Landing _Outcome\"
FROM SPACEXTABLE
WHERE \"Landing _Outcome\" = 'Success (drone ship)'
AND PayloadMass BETWEEN 4000 AND 6000
""", conn)

,Booster_Version,PayloadMass,Landing _Outcome
0,F9 FT B1022,4696,Success (drone ship)
1,F9 FT B1026,4600,Success (drone ship)
2,F9 FT B1021.2,5300,Success (drone ship)
3,F9 FT B1031.2,5200,Success (drone ship)


## 7. Total successful vs. failed mission outcomes

In [8]:
pd.read_sql_query("""
SELECT Mission_Outcome, COUNT(*) AS Count
FROM SPACEXTABLE
GROUP BY Mission_Outcome
""", conn)

,Mission_Outcome,Count
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


## 8. Booster(s) that carried the maximum payload

In [9]:
pd.read_sql_query("""
SELECT Booster_Version, PayloadMass
FROM SPACEXTABLE
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTABLE)
""", conn)

,Booster_Version,PayloadMass
0,F9 B5 B1048.4,15600
1,F9 B5 B1051.3,15600
2,F9 B5 B1056.4,15600
3,F9 B5 B1060.2,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1049.5,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1058.3,15600
8,F9 B5 B1049.4,15600
9,F9 B5 B1051.6,15600


## 9. Failed drone-ship landings in 2015 (booster + launch site)

In [10]:
pd.read_sql_query("""
SELECT Booster_Version, Launch_Site, \"Landing _Outcome\"
FROM SPACEXTABLE
WHERE \"Landing _Outcome\" = 'Failure (drone ship)'
AND Date LIKE '2015%'
""", conn)

,Booster_Version,Launch_Site,Landing _Outcome
0,F9 v1.1 B1015,CCAFS LC-40,Failure (drone ship)
1,F9 v1.1 B1012,CCAFS LC-40,Failure (drone ship)


## 10. Rank landing outcomes between 2010-06-04 and 2017-03-20 (descending count)

In [11]:
pd.read_sql_query("""
SELECT \"Landing _Outcome\", COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY \"Landing _Outcome\"
ORDER BY Outcome_Count DESC
""", conn)

,Landing _Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),6
2,Success (ground pad),5
3,Failure (drone ship),5
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Precluded (drone ship),1
7,Failure (parachute),1


In [12]:
conn.close()

## Summary

- Used SQL aggregate functions (SUM, AVG, MAX), filtering (LIKE, BETWEEN), and GROUP BY/ORDER BY to extract specific business insights directly from the launch dataset
- Confirmed NASA (CRS) total payload, F9 v1.1 average payload, first ground-pad landing date, and ranked landing outcome frequency
- These results corroborate the visual trends from Notebook 2 (EDA with Visualization)